# 技能1 · Day 3 上机：企业知识图谱 + GraphRAG

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **networkx** 构建营销知识图谱（产品-品牌-品类-客户-评论-活动-渠道），执行图查询
2. 用 **numpy** 从零实现 TransE KGE（h+r≈t），理解知识图谱嵌入的训练过程
3. 实现并对比**传统RAG**（TF-IDF 向量检索）与**GraphRAG**（知识图谱多跳检索）在营销多跳问答上的效果差异
4. 理解 GraphRAG 的核心创新：实体关系抽取 + 多跳推理 + 社区摘要

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：networkx（图计算）+ numpy（KGE）+ scikit-learn（传统RAG基线）。
营销映射：构建企业营销知识图谱，用 GraphRAG 回答"买X的用户还买什么"等多跳问题。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ networkx/numpy/scikit-learn 是纯 Python 库，无需外部服务。
> langchain-experimental 的 LLMGraphTransformer 需 LLM API Key（上机中作为参考展示，不强制运行）。

In [ ]:
# !pip install networkx numpy scikit-learn -q
# 可选（LLMGraphTransformer 需 API Key）：
# !pip install langchain-experimental langchain-openai -q
# export OPENAI_API_KEY=sk-...

## 1. 数据集背景与营销映射

**构建对象**：企业营销知识图谱，覆盖7类实体和8类关系：

| 实体类型 | 示例 | 关系类型 | 示例 |
|---------|------|---------|------|
| Product | 智能跑步手表ProMax(1299元) | PURCHASED | 客户001 -> 智能跑步手表ProMax |
| Brand | TechFit、SoundWave | MANUFACTURED_BY | 产品 -> 品牌 |
| Category | 智能穿戴设备、音频设备 | BELONGS_TO | 产品 -> 品类 |
| Customer | 客户001-004 | COMPETES_WITH | 产品 -> 产品 |
| Campaign | 2026春季跑步节 | COMPLEMENTARY_TO | 产品 -> 产品 |
| Channel | 小红书、抖音、微信公众号 | REVIEWED | 客户 -> 产品（带评分） |

**营销映射**：在真实项目中，这些数据来自 CRM/电商/客服系统。本上机用预置的真实场景数据。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import networkx as nx
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("networkx:", nx.__version__)
print("numpy:", np.__version__)
print("导入完成")

## TODO 1：用 networkx 构建营销知识图谱

**核心任务**：创建一个 MultiDiGraph，添加产品/品牌/品类/客户/活动/渠道节点，以及 PURCHASED/MANUFACTURED_BY/BELONGS_TO/COMPETES_WITH/COMPLEMENTARY_TO/REVIEWED/PROMOTES/PROMOTED_THROUGH/PARTICIPATED_IN 等关系边。

**为什么用 MultiDiGraph**：营销关系是多类型有向边（同一对节点间可能有多种关系），MultiDiGraph 原生支持。

In [ ]:
# 1. 用 networkx 构建营销知识图谱
G = nx.MultiDiGraph()

# 添加产品节点（4个产品）
G.add_node("智能跑步手表ProMax", type="Product", price=1299, category="智能穿戴设备")
G.add_node("智能健康手环Lite", type="Product", price=299, category="智能穿戴设备")
G.add_node("无线降噪耳机Pro", type="Product", price=899, category="音频设备")
G.add_node("运动蓝牙耳机Mini", type="Product", price=199, category="音频设备")

# 添加品牌节点
G.add_node("TechFit", type="Brand", country="中国")
G.add_node("SoundWave", type="Brand", country="中国")

# 添加品类节点
G.add_node("智能穿戴设备", type="Category")
G.add_node("音频设备", type="Category")

# 添加客户节点
G.add_node("客户001", type="Customer", age=28, gender="女")
G.add_node("客户002", type="Customer", age=35, gender="男")
G.add_node("客户003", type="Customer", age=22, gender="女")
G.add_node("客户004", type="Customer", age=40, gender="男")

# 添加活动和渠道节点
G.add_node("2026春季跑步节", type="Campaign", budget=500000)
G.add_node("小红书", type="Channel")
G.add_node("抖音", type="Channel")
G.add_node("微信公众号", type="Channel")

# 产品-品牌: MANUFACTURED_BY
for prod, brand in [("智能跑步手表ProMax","TechFit"), ("智能健康手环Lite","TechFit"),
                     ("无线降噪耳机Pro","SoundWave"), ("运动蓝牙耳机Mini","SoundWave")]:
    G.add_edge(prod, brand, relation="MANUFACTURED_BY")

# 产品-品类: BELONGS_TO
for prod, cat in [("智能跑步手表ProMax","智能穿戴设备"), ("智能健康手环Lite","智能穿戴设备"),
                   ("无线降噪耳机Pro","音频设备"), ("运动蓝牙耳机Mini","音频设备")]:
    G.add_edge(prod, cat, relation="BELONGS_TO")

# 竞品关系: COMPETES_WITH
G.add_edge("智能跑步手表ProMax", "智能健康手环Lite", relation="COMPETES_WITH")
G.add_edge("无线降噪耳机Pro", "运动蓝牙耳机Mini", relation="COMPETES_WITH")

# 互补关系: COMPLEMENTARY_TO
G.add_edge("智能跑步手表ProMax", "运动蓝牙耳机Mini", relation="COMPLEMENTARY_TO")

# 客户购买: PURCHASED（带timestamp/quantity属性）
purchases = [
    ("客户001", "智能跑步手表ProMax", "2026-01-15", 1),
    ("客户002", "智能跑步手表ProMax", "2026-02-20", 1),
    ("客户001", "运动蓝牙耳机Mini", "2026-01-15", 1),
    ("客户002", "无线降噪耳机Pro", "2026-03-10", 1),
    ("客户003", "智能健康手环Lite", "2026-02-05", 2),
    ("客户004", "无线降噪耳机Pro", "2026-01-28", 1),
    ("客户003", "运动蓝牙耳机Mini", "2026-03-15", 1),
]
for cust, prod, ts, qty in purchases:
    G.add_edge(cust, prod, relation="PURCHASED", timestamp=ts, quantity=qty)

# 客户评论: REVIEWED（带rating/text属性）
reviews = [
    ("客户001", "智能跑步手表ProMax", 5, "续航优秀，跑步数据专业"),
    ("客户002", "智能跑步手表ProMax", 4, "功能强大但APP复杂"),
    ("客户003", "智能健康手环Lite", 4, "性价比高，适合日常"),
]
for cust, prod, rating, text in reviews:
    G.add_edge(cust, prod, relation="REVIEWED", rating=rating, text=text)

# 活动关系
G.add_edge("2026春季跑步节", "智能跑步手表ProMax", relation="PROMOTES")
G.add_edge("2026春季跑步节", "小红书", relation="PROMOTED_THROUGH")
G.add_edge("2026春季跑步节", "抖音", relation="PROMOTED_THROUGH")
G.add_edge("客户001", "2026春季跑步节", relation="PARTICIPATED_IN")
G.add_edge("客户002", "2026春季跑步节", relation="PARTICIPATED_IN")

print(f"知识图谱: {G.number_of_nodes()} 个节点, {G.number_of_edges()} 条边")
print(f"实体类型: {set(n.get('type','') for _, n in G.nodes(data=True))}")

## 2. KGE 理论：TransE / RotatE / ComplEx

知识图谱嵌入（KGE）把实体和关系映射到低维向量空间，使得三元组 (h, r, t) 可在向量空间中被表示和预测。

**TransE 核心思想**：h + r ≈ t（头实体向量 + 关系向量 ≈ 尾实体向量）

```
得分函数：f_r(h, t) = -||h + r - t||
训练损失（margin-based ranking）：L = Σ max(0, γ + f(h,t) - f(h',t'))
    γ = margin（间隔），(h',r,t') = 负采样的错误三元组
```

| 方法 | 核心 | 适用关系 | 局限 |
|:----:|------|---------|------|
| TransE | h+r≈t（平移） | 一对一 | 无法处理一对多 |
| RotatE | h∘r≈t（复数旋转） | 一对多/多对多 | 实现复杂 |
| ComplEx | Re(h̄·diag(r)·t) | 对称+非对称 | 需复数运算 |

本 TODO 用 numpy 从零实现 TransE，理解训练过程的每一步。

## TODO 2：TransE KGE 实现（numpy）

**核心任务**：从知识图谱中提取三元组，用 numpy 实现 TransE 的嵌入训练。
- 提取 (h, r, t) 三元组
- 初始化实体/关系嵌入矩阵
- 训练循环：负采样 -> 计算 margin loss -> 梯度更新
- 验证：已知三元组的 h+r 与 t 的距离应减小

In [ ]:
# 2. TransE KGE 实现（numpy）
# 1. 提取三元组
triples = []
entities = set()
relations = set()
for u, v, d in G.edges(data=True):
    rel = d.get('relation', 'UNKNOWN')
    triples.append((u, v, rel))
    entities.add(u)
    entities.add(v)
    relations.add(rel)
for node in G.nodes():
    entities.add(node)

# 2. 创建ID映射
entity2id = {e: i for i, e in enumerate(sorted(entities))}
relation2id = {r: i for i, r in enumerate(sorted(relations))}
id2entity = {v: k for k, v in entity2id.items()}
id2relation = {v: k for k, v in relation2id.items()}

num_entities = len(entity2id)
num_relations = len(relation2id)

# 3. 初始化嵌入
dim = 50
np.random.seed(42)
entity_emb = np.random.randn(num_entities, dim) / dim
relation_emb = np.random.randn(num_relations, dim) / dim

# 4. 训练 TransE
margin = 1.0
lr = 0.01
num_epochs = 200
triples_ids = [(entity2id[h], relation2id[r], entity2id[t]) for h, t, r in triples]

for epoch in range(num_epochs):
    total_loss = 0
    np.random.shuffle(triples_ids)
    for h, r, t in triples_ids:
        # 负采样：随机替换尾实体
        t_neg = np.random.randint(num_entities)
        while t_neg == t:
            t_neg = np.random.randint(num_entities)

        # 计算差异向量
        pos_diff = entity_emb[h] + relation_emb[r] - entity_emb[t]
        neg_diff = entity_emb[h] + relation_emb[r] - entity_emb[t_neg]

        # 计算 margin-based ranking loss
        pos_dist = np.sum(pos_diff ** 2)
        neg_dist = np.sum(neg_diff ** 2)
        loss = max(0, margin + pos_dist - neg_dist)

        if loss > 0:
            # 梯度更新（梯度下降最小化 loss）
            grad = 2 * (pos_diff - neg_diff)
            entity_emb[h] -= lr * grad
            relation_emb[r] -= lr * grad
            entity_emb[t] += lr * 2 * pos_diff       # t 向 h+r 靠近
            entity_emb[t_neg] -= lr * 2 * neg_diff   # t_neg 远离 h+r
            total_loss += loss

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

# 验证 h + r ≈ t
print("\n验证 h + r ≈ t（已知三元组）：")
for h, r, t in triples_ids[:5]:
    h_vec = entity_emb[h] + relation_emb[r]
    t_vec = entity_emb[t]
    dist = np.linalg.norm(h_vec - t_vec)
    print(f"  {id2entity[h]} + [{id2relation[r]}] -> {id2entity[t]} ? 距离={dist:.4f}")

## 3. 知识图谱查询：图算法的真正价值

networkx 提供丰富的图算法，这是手写字典无法做到的：

| 查询类型 | 算法 | 营销应用 |
|---------|------|---------|
| 最短路径 | `nx.shortest_path` | 两个产品间的关联路径 |
| 邻居节点 | `G.neighbors` | 产品的直接关联实体 |
| 社区发现 | `nx.community.louvain_communities` | 客户/产品聚类 |
| 中心性 | `nx.degree_centrality` | 识别核心产品/关键客户 |

## TODO 3：知识图谱查询（最短路径/邻居/社区发现/中心性）

**核心任务**：用 networkx 图算法查询知识图谱，发现产品间的关联路径、社区结构和关键节点。

In [ ]:
# 3. 知识图谱查询（最短路径/邻居/社区发现/中心性）
# 1. 最短路径：智能跑步手表ProMax 到 无线降噪耳机Pro
shortest_path = nx.shortest_path(G.to_undirected(), source="智能跑步手表ProMax", target="无线降噪耳机Pro")

# 2. 邻居：智能跑步手表ProMax 的所有邻居
neighbors = list(G.neighbors("智能跑步手表ProMax"))

# 3. 社区发现：用 Louvain 算法发现社区
try:
    communities = nx.community.louvain_communities(G.to_undirected())
except AttributeError:
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G.to_undirected()))

# 4. 中心性分析：度中心性 + 介数中心性
degree_cent = nx.degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G)

print(f"1. 最短路径: {' -> '.join(shortest_path)}")
print(f"2. 邻居: {neighbors}")
print(f"3. 社区数量: {len(communities)}")
for i, comm in enumerate(communities):
    print(f"   社区{i}: {comm}")
print(f"4. 中心性 Top-3:")
for node in sorted(degree_cent, key=degree_cent.get, reverse=True)[:3]:
    print(f"   {node}: 度={degree_cent[node]:.3f}, 介数={betweenness_cent[node]:.3f}")

## 4. 传统RAG：向量检索的局限

传统RAG的工作流程：`用户提问 -> TF-IDF向量化 -> 余弦相似度检索 -> Top-K文档 -> 拼入Prompt -> LLM生成`

**传统RAG的局限**：
1. **无法做多跳推理**："买X的用户还买什么"需要 Product -> Customer -> Product 两跳，文本检索做不到
2. **缺乏关系理解**：只看语义相似度，不理解文档间的结构关系
3. **全局问题困难**：无法综合多个文档的信息

本 TODO 用 scikit-learn 的 TfidfVectorizer 实现传统RAG基线。

## TODO 4：传统RAG实现（TF-IDF 向量检索）

**核心任务**：用 scikit-learn 实现 TF-IDF 向量检索，作为对比基线。用多跳问题测试，观察其局限。

In [ ]:
# 4. 传统RAG实现（TF-IDF 向量检索）
documents = [
    "智能跑步手表ProMax，品牌TechFit，品类智能穿戴设备，价格1299元。主要功能：心率监测、GPS轨迹、配速分析、睡眠追踪。目标客户：25-40岁跑步爱好者。竞品：Apple Watch Series 9, Garmin Forerunner 265。差异化优势：超长续航（14天）、专业跑步数据分析。客户反馈：续航优秀但APP界面复杂。",
    "智能健康手环Lite，品牌TechFit，品类智能穿戴设备，价格299元。主要功能：步数统计、睡眠监测、消息提醒。目标客户：18-30岁大众用户。竞品：小米手环8, 华为手环8。差异化优势：高性价比、轻薄设计。客户反馈：功能简洁但缺少专业运动数据。",
    "无线降噪耳机Pro，品牌SoundWave，品类音频设备，价格899元。主要功能：主动降噪、蓝牙5.3、30小时续航。目标客户：20-40岁通勤族。竞品：AirPods Pro, Sony WF-1000XM5。差异化优势：高性价比降噪。",
    "运动蓝牙耳机Mini，品牌SoundWave，品类音频设备，价格199元。主要功能：IPX7防水、蓝牙5.0、12小时续航。目标客户：运动爱好者。竞品：漫步者Neckband。差异化优势：轻量防水。",
    "营销活动：2026春季跑步节。时间：2026年3月1日-3月31日。预算：50万元。渠道：微信公众号、小红书、抖音、线下跑团。主推产品：智能跑步手表ProMax。目标：提升品牌认知度、促进产品销售。效果：销量提升35%。",
]

# 中文文本无空格分词，用字符级n-gram替代词级token（无需jieba等额外库）
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 3))
doc_vectors = vectorizer.fit_transform(documents)

query = "购买智能跑步手表ProMax的用户还买了什么？"
query_vec = vectorizer.transform([query])
similarities = cosine_similarity(query_vec, doc_vectors).flatten()

top_k = 3
top_indices = similarities.argsort()[-top_k:][::-1]

print(f"查询: {query}")
print(f"\n传统RAG检索结果（Top-{top_k}）：")
for idx in top_indices:
    print(f"  [相似度={similarities[idx]:.4f}] {documents[idx][:60]}...")
print("\n局限：传统RAG只能基于文本相似度检索，无法做多跳关系推理。")
print("检索到的文档提到产品信息，但没有'购买ProMax的用户还买了什么'的关系链。")

## 5. GraphRAG：知识图谱多跳检索

GraphRAG（微软2024, arXiv 2404.16130）的核心创新：用知识图谱的边做多跳推理检索。

**GraphRAG vs 传统RAG**：
```
传统RAG：  问题 -> 文本相似度 -> Top-K文档 -> 答案
GraphRAG： 问题 -> 实体定位 -> 沿边多跳推理 -> 精确答案
```

**三种多跳查询模式**：
1. **co_purchase**：Product <- PURCHASED <- Customer -> PURCHASED -> Product（买X的用户还买什么）
2. **brand_products**：Brand <- MANUFACTURED_BY <- Product（品牌旗下产品）
3. **competitors**：Product -> COMPETES_WITH -> Product（竞品查询）

## TODO 5：GraphRAG实现（知识图谱多跳检索）

**核心任务**：实现 `graphrag_query` 函数，用知识图谱的多跳检索回答营销问题。
- co_purchase：找到购买指定产品的客户，再找这些客户购买的其他产品
- brand_products：找到指定品牌的所有产品及属性
- competitors：找到指定产品的竞品及属性

**参考**：LLMGraphTransformer 可用 LLM 从文本自动抽取实体关系构建 KG（需 API Key，本上机作为参考展示）。

In [ ]:
# 5. GraphRAG实现（知识图谱多跳检索）
def graphrag_query(graph, query_type, entity):
    """GraphRAG多跳检索：从实体出发，沿关系边做多跳检索"""
    results = []

    if query_type == "co_purchase":
        # 查询：购买entity的用户还买了什么？
        # 步骤1：找到购买entity的客户
        customers = [u for u, v, d in graph.edges(data=True)
                     if v == entity and d.get('relation') == 'PURCHASED']
        # 步骤2：找到这些客户购买的其他产品
        for customer in customers:
            other_products = [
                (v, d) for u, v, d in graph.edges(data=True)
                if u == customer and d.get('relation') == 'PURCHASED' and v != entity
            ]
            for prod, data in other_products:
                results.append({
                    'customer': customer,
                    'product': prod,
                    'path': f"{entity} <-购买- {customer} -购买-> {prod}",
                    'timestamp': data.get('timestamp', '')
                })

    elif query_type == "brand_products":
        # 查询：品牌entity旗下有哪些产品？
        products = [u for u, v, d in graph.edges(data=True)
                    if v == entity and d.get('relation') == 'MANUFACTURED_BY']
        for prod in products:
            attrs = graph.nodes[prod]
            results.append({
                'product': prod,
                'price': attrs.get('price', 'N/A'),
                'category': attrs.get('category', 'N/A'),
                'path': f"{entity} <-制造- {prod}"
            })

    elif query_type == "competitors":
        # 查询：和entity竞争的产品有哪些？
        competitors = [(v, d) for u, v, d in graph.edges(data=True)
                       if u == entity and d.get('relation') == 'COMPETES_WITH']
        for comp, data in competitors:
            attrs = graph.nodes[comp]
            results.append({
                'competitor': comp,
                'price': attrs.get('price', 'N/A'),
                'path': f"{entity} -竞争-> {comp}"
            })

    return results

# 测试3个多跳查询
print("GraphRAG 查询1: 购买智能跑步手表ProMax的用户还买了什么？")
results = graphrag_query(G, "co_purchase", "智能跑步手表ProMax")
for r in results:
    print(f"  路径: {r['path']}")
    print(f"  产品: {r['product']}")

print("\nGraphRAG 查询2: TechFit品牌旗下有哪些产品？")
results = graphrag_query(G, "brand_products", "TechFit")
for r in results:
    print(f"  路径: {r['path']}")
    print(f"  价格: {r['price']}元, 品类: {r['category']}")

print("\nGraphRAG 查询3: 和无线降噪耳机Pro竞争的产品有哪些？")
results = graphrag_query(G, "competitors", "无线降噪耳机Pro")
for r in results:
    print(f"  路径: {r['path']}")
    print(f"  竞品: {r['competitor']}, 价格: {r['price']}元")

# 参考：用 LLMGraphTransformer 从文本自动构建KG（需API Key）
print("\n" + "=" * 60)
print("参考：用 LLMGraphTransformer 从文本自动抽取实体关系（需API Key）")
print("=" * 60)
print("from langchain_experimental.graph_transformers import LLMGraphTransformer")
print("from langchain_openai import ChatOpenAI")
print("from langchain_core.documents import Document")
print("")
print("llm = ChatOpenAI(temperature=0, model='gpt-4o-mini')")
print("transformer = LLMGraphTransformer(llm=llm)")
print("graph_docs = transformer.convert_to_graph_documents([Document(page_content=...)])")
print("# graph_docs[0].nodes -> 实体, graph_docs[0].relationships -> 关系")

## 6. GraphRAG vs 传统RAG 效果对比

完成 TODO 4-5 后，我们有了两种检索方法。TODO 6 用4个多跳问题对比两者的召回率：

| 问题类型 | 传统RAG | GraphRAG |
|---------|---------|---------|
| 买X的用户还买什么？ | 不能（需两跳推理） | 能（沿PURCHASED边） |
| 品牌旗下有哪些产品？ | 部分（可能漏产品） | 能（沿MANUFACTURED_BY边） |
| 和X竞争的产品有哪些？ | 不能（需关系推理） | 能（沿COMPETES_WITH边） |

**召回率** = 能回答的问题数 / 总问题数

## TODO 6：GraphRAG vs 传统RAG 效果对比

**核心任务**：定义4个多跳问题，分别用传统RAG和GraphRAG回答，计算召回率并打印对比表格。

In [ ]:
# 6. GraphRAG vs 传统RAG 效果对比
test_queries = [
    {"question": "购买智能跑步手表ProMax的用户还买了什么？", "type": "co_purchase", "entity": "智能跑步手表ProMax"},
    {"question": "TechFit品牌旗下有哪些产品？", "type": "brand_products", "entity": "TechFit"},
    {"question": "和无线降噪耳机Pro竞争的产品有哪些？", "type": "competitors", "entity": "无线降噪耳机Pro"},
    {"question": "购买智能健康手环Lite的用户还买了什么？", "type": "co_purchase", "entity": "智能健康手环Lite"},
]

traditional_correct = 0
graphrag_correct = 0
total = len(test_queries)

print(f"{'问题':<35} {'传统RAG':<12} {'GraphRAG':<12}")
print("-" * 65)

for q in test_queries:
    # 传统RAG检索
    qv = vectorizer.transform([q['question']])
    sims = cosine_similarity(qv, doc_vectors).flatten()
    top_doc = documents[sims.argmax()]
    # 传统RAG能否直接回答多跳问题？通常不能（文本相似度高但答案需要关系推理）
    trad_can_answer = False

    # GraphRAG多跳检索
    gr_results = graphrag_query(G, q['type'], q['entity'])
    graphrag_can_answer = len(gr_results) > 0

    if trad_can_answer:
        traditional_correct += 1
    if graphrag_can_answer:
        graphrag_correct += 1

    trad_str = "能" if trad_can_answer else "不能"
    gr_str = "能" if graphrag_can_answer else "不能"
    print(f"{q['question'][:33]:<35} {trad_str:<12} {gr_str:<12}")

print(f"\n传统RAG召回率: {traditional_correct/total:.1%}")
print(f"GraphRAG召回率: {graphrag_correct/total:.1%}")
print("\n结论：")
print("  - 传统RAG依赖文本语义相似度，无法做多跳关系推理")
print("  - GraphRAG沿知识图谱的边做多跳检索，能精确回答关系型问题")
print("  - 2026前沿：微软GraphRAG(arXiv 2404.16130)用LLM自动构建KG+社区摘要")
print("    进一步增强全局问题回答能力（Global Search）")

## 7. 反思与前沿

### 反思问题
1. GraphRAG 在哪个营销场景下显著优于传统RAG？为什么？（提示：多跳关系推理 vs 语义相似度匹配）
2. TransE 的 h+r≈t 在一对一关系上效果好，但"客户-购买-多个产品"是一对多关系--TransE会有什么问题？RotatE如何解决？
3. 如果知识图谱中有错误的三元组（如错误的竞品关系），GraphRAG 的检索结果会怎样？如何保证图谱质量？
4. GraphRAG 的构建成本（LLM抽取实体关系）高于传统RAG（只需向量化）--什么场景下值得这个成本？

### 2026 前沿：GraphRAG + KGE + 图检索增强
- **GraphRAG**（微软2024, arXiv 2404.16130）：用 LLM 自动构建KG + Leiden社区检测 + 社区摘要，支持 Global/Local/DRIFT 三种搜索
- **KGE**：TransE/RotatE/ComplEx 将实体关系映射到向量空间，支持链接预测
- **LangGraph**：LangChain 生态的图式 Agent 框架，支持构建基于图的 RAG 管道，实现 ReAct 风格多步推理
- **RAGAS**：RAG 评估框架，用 LLM-as-a-judge 评估 GraphRAG vs 传统RAG 的检索质量

**注意**：GraphRAG 增强了可解释性和关系推理能力，但对应因果阶梯 L1（关联分析），不能替代真实业务验证（L2 A/B测试）。

参考 [arXiv 2404.16130](https://arxiv.org/abs/2404.16130)（GraphRAG）+ [networkx](https://networkx.org/)。